# **LiminalGPT Stage 2**: _Self attention_

A basic transformer based neural network trained on a literary text corpus.

In Stage 2, we explore the concept of self attention in a transformer architecture.

LiminalGPT is based on [Vaswani et al. (2017)](https://arxiv.org/pdf/1706.03762).


In [113]:
import torch
from typing import Final

SEED: Final[int] = 3433

torch.manual_seed(SEED);

## Builiding the intuition behind self attention

We will compute a running average of feature vectors for each token position. For a token at position `i`, we average the features from all tokens at positions `0` through `i` (inclusive). This creates a representation that captures the "context" or "history" leading up to each token.

Given a tensor of shape `(B, T, C)` where:

- `B` = batch size
- `T` = sequence length (time steps)
- `C` = number of channels (feature dimensions)

For each token, we want to aggregate information only from previous tokens and itself (not future tokens).


### Version 1: weighted aggregation via manual loops


In [114]:
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

For now, let us perform the weighted aggregation for each token via loops:


In [115]:
xbow = torch.zeros((B, T, C))

for b in range(B):
    for t in range(T):
        # extract running history of tokens
        xprev = x[b, : t + 1]  # shape (T, C)
        # calculate running mean
        xbow[b, t] = xprev.mean(0)

In [116]:
x[0]

tensor([[-1.4970, -0.6357],
        [-0.6095, -0.1586],
        [ 0.9341, -0.0361],
        [-0.4688, -0.0288],
        [-1.2407,  0.2724],
        [ 0.1070, -1.9325],
        [ 0.8531, -0.5161],
        [ 0.0290, -0.1809]])

In [117]:
xbow[0]

tensor([[-1.4970, -0.6357],
        [-1.0532, -0.3972],
        [-0.3908, -0.2768],
        [-0.4103, -0.2148],
        [-0.5764, -0.1174],
        [-0.4625, -0.4199],
        [-0.2745, -0.4336],
        [-0.2366, -0.4020]])

### Version 2: weighted aggregation via matrix multiplication


#### 2.a Calculating global mean


In [118]:
a = torch.ones(3, 3)
print(f"{a = }")

b = torch.randint(0, 10, (3, 2)).float()
print(f"\n{b = }")

c = a @ b
print(f"\n{c = }")

mean = c.mean(0)
print(f"\n{mean = }")

a = tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])

b = tensor([[1., 4.],
        [4., 9.],
        [0., 4.]])

c = tensor([[ 5., 17.],
        [ 5., 17.],
        [ 5., 17.]])

mean = tensor([ 5., 17.])


#### 2.b Calculating running mean via `torch.tril()`


[torch.tril()](https://docs.pytorch.org/docs/stable/generated/torch.tril.html) converts all elements above the diagonal to 0:


In [119]:
ex = torch.ones(3, 3)
print(f"before tril():\n{ex}")
print("\n")
ex = torch.tril(ex)
print(f"after tril():\n{ex}")

before tril():
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])


after tril():
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])


We can use `tril()` in matrix multiplication to calculate the running mean:


In [120]:
a = torch.tril(torch.ones(3, 3))
print(f"{a = }")

b = torch.randint(0, 10, (3, 2)).float()
print(f"\n{b = }")

c = a @ b
print(f"\n{c = }")

a = tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

b = tensor([[9., 1.],
        [5., 9.],
        [8., 4.]])

c = tensor([[ 9.,  1.],
        [14., 10.],
        [22., 14.]])


But how do we calculate the mean from the running sums?

We can perform a mean by **normalizing the rows** of tensor `a`:


In [121]:
a = a / a.sum(1, True)
a

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

We know:

$$mean = \frac{1}{\text{total\ elements}}\times\text{sum of elements} $$

For a running average, we want each row of `a` to be weights that sum to 1, so the result is the average and not the sum.

In the example above:

Row 0: [1.0, 0, 0] → sums to 1 → averages token 0 only

Row 1: [0.5, 0.5, 0] → sums to 1 → averages tokens 0-1

Row 2: [0.33, 0.33, 0.33] → sums to 1 → averages tokens 0-2

Now when we do `a @ b`, we get the running mean instead of the running sum:

- Position 0:
  $$1.0 × b[0] = \textcolor{lightblue}{\frac{1}{1} \times (b[0])} =  \text{just b[0]}$$

- Position 1:
  $$0.5 × b[0] + 0.5 × b[1] = \textcolor{lightblue}{\frac{1}{2} \times (b[0] + b[1])} = \text{mean of } b[0] \text{ and } b[1]$$

- Position 2:
  $$0.33 × b[0] + 0.33 × b[1] + 0.33 × b[2] = \textcolor{lightblue}{\frac{1}{3} \times (b[0] + b[1] + b[2])} = \text{mean of all three}$$

Observe how these $\textcolor{lightblue}{\text{expressions}}$ match our _mean_ formula!

Therefore, if we normalize the weights so that they sum to 1, we transform the matrix multiplication from computing a weighted sum into computing a **weighted average**. This is exactly what our manual loop implementation was trying to do!


#### 2.c Complete vectorized method


In [122]:
# =version 1
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))

for b in range(B):
    for t in range(T):
        # extract running history of tokens
        xprev = x[b, : t + 1]  # shape (T, C)
        # calculate running mean
        xbow[b, t] = xprev.mean(0)

In [123]:
# version 2
weights = torch.tril(torch.ones(T, T))  # reason for shape explained below
weights = weights / weights.sum(1, True)
weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [124]:
#  (T, T) @ (B,T,C) --> (B, T, T) @ (B, T, C) --> (B,T,C)
# we want the output to have same shape as x so weights shape must be (T,T)
xbow2 = weights @ x
print("xbow and xbow2 have close values:", torch.allclose(xbow, xbow2))
# compare batch 1
print("\nfirst batch of xbow and xbow2:")
xbow[0], xbow2[0]

xbow and xbow2 have close values: True

first batch of xbow and xbow2:


(tensor([[ 0.2356,  0.1581],
         [ 0.6854, -0.1624],
         [ 0.5331, -0.1489],
         [ 0.5530,  0.1019],
         [ 0.6846, -0.0534],
         [ 0.5232,  0.0711],
         [ 0.5349, -0.0523],
         [ 0.5083,  0.0794]]),
 tensor([[ 0.2356,  0.1581],
         [ 0.6854, -0.1624],
         [ 0.5331, -0.1489],
         [ 0.5530,  0.1019],
         [ 0.6846, -0.0534],
         [ 0.5232,  0.0711],
         [ 0.5349, -0.0523],
         [ 0.5083,  0.0794]]))

### Version 3: weighted aggregation via softmax


Read about [torch.Tensor.masked_fill()](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.masked_fill_.html).

This version achieves the same aggregation effect but helps us reason about the mechanism in a different pov.

---

In all examples, we intitialized weight matrix to zero to demonstrate the mechanics of aggregation.

In practice, these weights are data-dependent and learned from the input.
We can think of their magnitude as interaction strength or affinity between tokens:

- Higher weights mean stronger connections between token pairs
- Lower weights mean weaker connections

Let sequence: ["The", "cat", "sat"]

For this sequence we would have a 3x3 weight matrix with one entry for every possible pair of tokens in the sequence:

- w[0,0]: pair ("The" → "The") - how much "The" attends to itself
- w[1,0]: pair ("cat" → "The") - how much "cat" attends to "The"
- w[1,2]: pair ("cat" → "sat") - how much "cat" attends to "sat"

The weights determine how much each token _pays attention_ to other tokens. :)

---

The masked fill operation can be thought of as a clamping operation that ensures tokens from the future cannot communicate/aggregate.

By setting future positions to -inf before softmax, we enforce causality:

- Token at position t can only attend to tokens at positions 0 through t (including itself)
- Token at position t cannot "see" tokens at positions t+1, t+2, ..., T-1

This is crucial for autoregressive models (like GPT) where we generate tokens sequentially.
During training, we process all tokens in parallel, but we must prevent information leakage from future tokens.
The mask ensures that the model learns to predict each token using only the context that came before it.


In [ ]:
tril = torch.tril(torch.ones(T, T))

weights = torch.zeros(T, T)
weights = weights.masked_fill(tril == 0, float("-inf"))
weights

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

Softmax-ing along dim 1 gives us the same weight tensor as in version 2:

Softmax exponentiates each element and divides each element by the sum of all elements. Here:

- $\exp(-\infty) = 0$
- $\exp(0) = 1$


In [126]:
weights = weights.softmax(1)
weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [ ]:
# same aggregation as v2 via matrix mul
xbow3 = weights @ x

print("xbow and xbow3 have close values:", torch.allclose(xbow, xbow3))
# compare batch 1
print("\nfirst batch of xbow and xbow3:")
xbow[0], xbow3[0]

xbow and xbow3 have close values: True

first batch of xbow and xbow3:


(tensor([[ 0.2356,  0.1581],
         [ 0.6854, -0.1624],
         [ 0.5331, -0.1489],
         [ 0.5530,  0.1019],
         [ 0.6846, -0.0534],
         [ 0.5232,  0.0711],
         [ 0.5349, -0.0523],
         [ 0.5083,  0.0794]]),
 tensor([[ 0.2356,  0.1581],
         [ 0.6854, -0.1624],
         [ 0.5331, -0.1489],
         [ 0.5530,  0.1019],
         [ 0.6846, -0.0534],
         [ 0.5232,  0.0711],
         [ 0.5349, -0.0523],
         [ 0.5083,  0.0794]]))

## Building self attention


In [ ]:
import torch.nn as nn

B, T, C = 4, 8, 32

x = torch.randn(B, T, C)

# single Head performing self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)

k, q = key(x), query(x)  # (B, T, 16)